# Alpha Vantage — ETF Sector Lookthrough

Given a ticker, fetch its sector breakdown via the Alpha Vantage `ETF_PROFILE` endpoint.

API key is loaded from `configs/portfolio_monitor/local.env` (`ALPHA_VANTAGE_API_KEY`).

In [1]:
import os
import sys
from pathlib import Path

# Ensure repo root is on sys.path when running from notebooks/
repo_root = Path("__file__").resolve().parents[2]
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Load local.env so ALPHA_VANTAGE_API_KEY is available
from dotenv import load_dotenv
load_dotenv(repo_root / "configs" / "portfolio_monitor" / "local.env", override=True)

ALPHA_VANTAGE_API_KEY = os.environ["ALPHA_VANTAGE_API_KEY"]
print(f"API key loaded: {ALPHA_VANTAGE_API_KEY[:4]}****")

API key loaded: CT1I****


In [ ]:
# ── Configuration ────────────────────────────────────────────────
TICKER = "SPY"   # ← change this to any ETF ticker
# ─────────────────────────────────────────────────────────────────

In [6]:
from market_helper.data_sources.alpha_vantage import AlphaVantageClient

client = AlphaVantageClient(api_key=ALPHA_VANTAGE_API_KEY, request_spacing_seconds=0)
weights = client.fetch_etf_sector_weightings(TICKER)
print(f"Fetched {len(weights)} sector rows for {TICKER}")

AlphaVantageClientError: Alpha Vantage ETF profile for ACWD did not include sectors

In [7]:
import pandas as pd

df = (
    pd.DataFrame([{"sector": w.sector, "weight": w.weight} for w in weights])
    .sort_values("weight", ascending=False)
    .reset_index(drop=True)
)
df["weight_pct"] = (df["weight"] * 100).round(2).astype(str) + "%"
print(f"\n{TICKER} — Sector Breakdown")
print(df[["sector", "weight_pct"]].to_string(index=False))


ACWD — Sector Breakdown
                sector weight_pct
INFORMATION TECHNOLOGY      33.0%
            FINANCIALS      10.7%
COMMUNICATION SERVICES      10.3%
CONSUMER DISCRETIONARY      10.0%
            HEALTHCARE       9.3%
           INDUSTRIALS       9.1%
      CONSUMER STAPLES       5.1%
                ENERGY       4.0%
             UTILITIES       2.5%
             MATERIALS       1.9%


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(df["sector"][::-1], df["weight"][::-1] * 100)
ax.set_xlabel("Weight (%)")
ax.set_title(f"{TICKER} — Sector Breakdown")
plt.tight_layout()
plt.show()